In [17]:
# Open clinvar.vcf as plain text

with open("clinvar.vcf", "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        # Skip metadata lines
        if line.startswith("#"):
            continue
        
        # Split tab-separated columns
        columns = line.strip().split("\t")
        
        chrom = columns[0]
        pos = columns[1]
        ref = columns[3]
        alt = columns[4]
        info = columns[7]
        
        print("Chrom:", chrom)
        print("Position:", pos)
        print("Ref:", ref)
        print("Alt:", alt)
        print("Info:", info)
        print("-" * 50)
        
        break  

Chrom: 1
Position: 66926
Ref: AG
Alt: A
Info: ALLELEID=3544463;CLNDISDB=Human_Phenotype_Ontology:HP:0000547,MONDO:MONDO:0019200,MeSH:D012174,MedGen:C0035334,OMIM:268000,OMIM:PS268000,Orphanet:791;CLNDN=Retinitis_pigmentosa;CLNHGVS=NC_000001.11:g.66927del;CLNREVSTAT=criteria_provided,_single_submitter;CLNSIG=Uncertain_significance;CLNSIGSCV=SCV005419006;CLNVC=Deletion;CLNVCSO=SO:0000159;GENEINFO=OR4F5:79501;MC=SO:0001627|intron_variant;ORIGIN=0
--------------------------------------------------


In [18]:
splice_variants = []

with open("clinvar.vcf", "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        if line.startswith("#"):
            continue
        
        columns = line.strip().split("\t")
        chrom = columns[0]
        pos = columns[1]
        ref = columns[3]
        alt = columns[4]
        info = columns[7].lower()
        
        # Extract molecular consequence (MC field)
        if "mc=" in info:
            mc_field = ""
            for field in info.split(";"):
                if field.startswith("mc="):
                    mc_field = field
                    break
            
            if "splice_donor_variant" in mc_field or \
               "splice_acceptor_variant" in mc_field or \
               "splice_region_variant" in mc_field:
                
                # Extract clinical significance
                clnsig = "unknown"
                for field in info.split(";"):
                    if field.startswith("clnsig="):
                        clnsig = field.split("=")[1]
                        break
                
                splice_variants.append((chrom, pos, ref, alt, clnsig))

print("Total true splice variants:", len(splice_variants))

Total true splice variants: 107024


In [19]:
from collections import Counter

# Count clinical significance categories
clnsig_counter = Counter()

for var in splice_variants:
    clnsig_counter[var[4]] += 1

print("=== DATASET STATISTICS ===")
print("Total splice variants:", len(splice_variants))
print()

print("Clinical Significance Distribution:")
for key, value in clnsig_counter.items():
    print(f"{key}: {value}")

=== DATASET STATISTICS ===
Total splice variants: 107024

Clinical Significance Distribution:
uncertain_significance: 10808
benign: 252
likely_benign: 891
unknown: 44186
benign/likely_benign: 87
pathogenic: 15154
pathogenic/likely_pathogenic: 4747
likely_pathogenic: 28443
not_provided: 281
conflicting_classifications_of_pathogenicity: 2014
no_classifications_from_unflagged_records: 27
no_classification_for_the_single_variant: 49
drug_response: 11
pathogenic,_low_penetrance: 24
likely_pathogenic,_low_penetrance: 1
risk_factor: 17
affects: 5
likely_risk_allele: 8
other: 6
likely_pathogenic/likely_pathogenic,_low_penetrance: 1
likely_pathogenic/pathogenic,_low_penetrance: 1
pathogenic|other: 1
pathogenic|association: 1
association|drug_response|risk_factor: 1
association: 6
pathogenic|affects: 1
likely_benign|drug_response|other: 1


In [20]:
positive = []
negative = []
others = []

for var in splice_variants:
    clnsig = var[4]
    
    if "pathogenic" in clnsig:
        positive.append(var)
    elif "benign" in clnsig:
        negative.append(var)
    else:
        others.append(var)

print("\n=== BINARY CLASS DISTRIBUTION ===")
print("Pathogenic:", len(positive))
print("Benign:", len(negative))
print("Other (VUS/conflicting/etc.):", len(others))

print("\nClass Balance (%):")
total_binary = len(positive) + len(negative)
if total_binary > 0:
    print("Pathogenic %:", round(len(positive)/total_binary*100, 2))
    print("Benign %:", round(len(negative)/total_binary*100, 2))


=== BINARY CLASS DISTRIBUTION ===
Pathogenic: 50388
Benign: 1231
Other (VUS/conflicting/etc.): 55405

Class Balance (%):
Pathogenic %: 97.62
Benign %: 2.38


In [21]:
clean_dataset = []

for var in splice_variants:
    chrom, pos, ref, alt, clnsig = var
    
    if "pathogenic" in clnsig:
        label = 1
        clean_dataset.append((chrom, pos, ref, alt, label))
    
    elif "benign" in clnsig:
        label = 0
        clean_dataset.append((chrom, pos, ref, alt, label))

print("Total clean labeled variants:", len(clean_dataset))

Total clean labeled variants: 51619


In [22]:
import csv

with open("splice_dataset_full.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["chrom", "position", "ref", "alt", "label"])
    
    for row in clean_dataset:
        writer.writerow(row)

print(" CSV file saved as splice_dataset_full.csv")

 CSV file saved as splice_dataset_full.csv


In [23]:
import pandas as pd

df = pd.read_csv("splice_dataset_full.csv")
print(df.head())
print("Total rows:", len(df))

  chrom  position                                                ref  \
0     1    939398                                                  G   
1     1    939398                                                  G   
2     1    939398                                                  G   
3     1    939398                                                  G   
4     1    939398  GCCTCCCCAGCCACGGTGAGGACCCACCCTGGCATGATCCCCCTCATCA   

                                                 alt  label  
0  GCCTCCCCAGCCACGGTGAGGACCCACCCTGGCATGATCCCCCTCATCA      0  
1  GCCTCCCCAGCCACGGTGAGGACCCACCCTGGCATGATCCCCCTCA...      0  
2  GCCTCCCCAGCCACGGTGAGGACCCACCCTGGCATGATCCCCCTCA...      0  
3  GCCTCCCCAGCCACGGTGAGGACCCACCCTGGCATGATCCCCCTCA...      0  
4                                                  G      0  
Total rows: 51619


In [24]:
import pandas as pd

df = pd.read_csv("splice_dataset_full.csv")

print("Before duplicate removal:", len(df))

Before duplicate removal: 51619


In [25]:
df_unique = df.drop_duplicates(subset=["chrom", "position", "ref", "alt"])

print("After duplicate removal:", len(df_unique))

After duplicate removal: 51619


In [26]:
df_unique.to_csv("splice_dataset_unique.csv", index=False)

print("Clean dataset saved as splice_dataset_unique.csv")

Clean dataset saved as splice_dataset_unique.csv


In [27]:
duplicates_removed = len(df) - len(df_unique)
print("Duplicates removed:", duplicates_removed)

Duplicates removed: 0
